<a href="https://colab.research.google.com/github/anparashar/HDRNet/blob/main/HRDNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wandb -qU

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.3/184.3 kB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.5/206.5 kB 14.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 8.1 MB/s eta 0:00:00


In [ ]:
#import packages
import os
import cv2
from pathlib import Path
a='/content/drive/MyDrive/180x180 Image/'
p=Path(a)
i=0
listPaths=list(p.glob('*.jpg'))
for path in listPaths:
    img_g= cv2.imread(str(path))  
    #green_channel_g = img_g[:,:,1] 
    img_resize = cv2.resize( img_g, (512,512) )
    #img_resize=getGaussianFilter(img_resize,size,sigma)
    cv2.imwrite('/content/drive/MyDrive/train fundus/original/'+str(i)+'.jpg',img_resize)
    i=i+1

#default number of epochs
epochs=50
 
#default batch size for training
batch_size=128
 
#noise-level for training
sigma=25.0  #change it according to noise level in your dataset

#genDataPath='/content/drive/MyDrive/NORMALupdate2'
genDataPath='/content/drive/MyDrive/180x180 Image/'

#path to save the genPatches
save_dir='trainingPatch'
 
#path to training data
data='/content/trainingPatch/img_clean_pats.npy'
 
#variables required to generate patch
pat_size=40
stride=10
step=0

In [ ]:
# import the libraries
import os
import numpy as np
import cv2
 
from pathlib import Path
 
# define scales
scales=[1,0.9,0.8,0.7]
#scales=[1,0.9,0.8]
count=0
 
# create the imageArrays
p=Path(genDataPath)
 
listPaths=list(p.glob('./*.jpg'))
#print(listPaths)
imgArray=[]
for path in listPaths:
    imgArray.append(cv2.imread(str(path),0))
print('lenImages',len(imgArray))
 
#calculate the number of patches
for i in range(len(imgArray)):
    img = imgArray[i] 
    for s in range(len(scales)):
        newsize=(int(img.shape[0]*scales[s]),int(img.shape[1]*scales[s]))
        img_s=cv2.resize(img,newsize,interpolation=cv2.INTER_CUBIC)
        im_h,im_w=img_s.shape
        for x in range(0+step,(im_h-pat_size),
                      stride):
            for y in range(0+step,(im_w-pat_size),
                    stride):
                count +=1
 
origin_patch_num=count
if origin_patch_num % batch_size !=0:
    numPatches=(origin_patch_num/batch_size +1)*batch_size
else:
    numPatches=origin_patch_num
print('total patches=%d, batch_size=%d, total_batches=%d' % 
        (numPatches, batch_size, numPatches/batch_size))
 
#numpy array to contain patches for training
inputs=np.zeros((int(numPatches), int(pat_size), int(pat_size),1),dtype=np.uint8)
 
#generate patches
count=0
for i in range(len(imgArray)):
    img=imgArray[i]
    for s in range(len(scales)):
        newsize=(int(img.shape[0]*scales[s]),int(img.shape[1]*scales[s]))
        img_s=cv2.resize(img,newsize,interpolation=cv2.INTER_CUBIC)
        img_s=np.reshape(np.array(img_s,dtype="uint8"),
                (img_s.shape[0],img_s.shape[1],1))
        im_h,im_w, _ = img_s.shape
        for x in range(0+step,im_h-pat_size,stride):
            for y in range(0+step,im_w-pat_size,
                           stride):
                inputs[count,:,:,:]=img_s[x:x+pat_size,
                    y:y+pat_size,:]
                
                count += 1
                
 
 
#pad the batch
if count < numPatches:
    to_pad=int(numPatches-count)
    inputs[-to_pad:,:,:,:]=inputs[:to_pad,:,:,:]
 
if not os.path.exists(save_dir):
    os.mkdir(save_dir)
    np.save(os.path.join(save_dir,"img_clean_pats"),inputs)

lenImages 401
total patches=227495, batch_size=128, total_batches=1777


In [ ]:
import wandb
wandb.login()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
## model with conv2d, dialated conv2d and separable conv2d layer

from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Conv2D,SeparableConv2D
from keras.layers import BatchNormalization
from keras.layers import Activation
from keras import optimizers
import keras.backend as K
import numpy as np
from tensorflow import keras
import pandas as pd
from keras.callbacks import LearningRateScheduler

# create CNN model
model=Sequential()

model.add(Conv2D(64,(3,3),padding="same",input_shape=(None,None,1)))
model.add(Activation('relu'))
#model.add(MultiResBlock(64, input))
for layers in range(2,6):
    model.add(Conv2D(64,(3,3),padding="same"))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv2D(64,(3,3),padding="same", dilation_rate=2))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(SeparableConv2D(64,(3,3),padding="same"))
    model.add(BatchNormalization())
    model.add(Activation('relu'))

model.add(Conv2D(1,(3,3),padding="same"))
model.summary()
 


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, None, None, 64)    640       
                                                                 
 activation (Activation)     (None, None, None, 64)    0         
                                                                 
 conv2d_1 (Conv2D)           (None, None, None, 64)    36928     
                                                                 
 batch_normalization (BatchN  (None, None, None, 64)   256       
 ormalization)                                                   
                                                                 
 activation_1 (Activation)   (None, None, None, 64)    0         
                                                                 
 conv2d_2 (Conv2D)           (None, None, None, 64)    36928     
                                                        

In [ ]:
# load the data and normalize it

cleanImages=np.load(data)
cleanImages=cleanImages/255.0
cleanImages=cleanImages.astype('float32')

total_runs=steps_per_epoch=len(cleanImages)//batch_size
 
 
# define augmentor and create custom flow
aug = ImageDataGenerator(rotation_range=30, fill_mode="nearest")
def myFlow(generator,X):
      for batch in generator.flow(x=X,batch_size=batch_size,seed=7):
        trueNoiseBatch=np.random.normal(0.0,10/255,batch.shape)
        noisyImagesBatch=batch+trueNoiseBatch
        yield (noisyImagesBatch,trueNoiseBatch)
 
 
# create custom loss, compile the model
print("[INFO] compilingTheModel")

def custom_loss(y_true,y_pred):
    diff=y_true-y_pred
    res=K.sum(diff*diff)/(2*batch_size)
    return res

def lr_decay(epoch):
    initAlpha=0.001
    factor=0.5
    dropEvery=5
    alpha=initAlpha*(factor ** np.floor((1+epoch)/dropEvery))
    return float(alpha)
callbacks=[LearningRateScheduler(lr_decay)]
opt=keras.optimizers.Adam(learning_rate=0.001)
  
model.compile(loss=custom_loss,optimizer=opt)
#
print(len(cleanImages))


# train
model.fit(myFlow(aug,cleanImages),epochs=epochs,steps_per_epoch=len(cleanImages)//batch_size,callbacks=callbacks,verbose=1)


# save the model
model.save('myModel_hybrid_layer.h5')

[INFO] compilingTheModel
227495
Epoch 1/50
1777/1777 [==============================] - 360s 192ms/step - loss: 2.1739 - lr: 0.0010
Epoch 2/50
1777/1777 [==============================] - 342s 192ms/step - loss: 0.3389 - lr: 0.0010
Epoch 3/50
1777/1777 [==============================] - 342s 192ms/step - loss: 0.2385 - lr: 0.0010
Epoch 4/50
1777/1777 [==============================] - 342s 192ms/step - loss: 0.2054 - lr: 0.0010
Epoch 5/50
1777/1777 [==============================] - 342s 193ms/step - loss: 0.1505 - lr: 5.0000e-04
Epoch 6/50
1777/1777 [==============================] - 342s 192ms/step - loss: 0.1502 - lr: 5.0000e-04
Epoch 7/50
1777/1777 [==============================] - 342s 192ms/step - loss: 0.1451 - lr: 5.0000e-04
Epoch 8/50
1777/1777 [==============================] - 342s 193ms/step - loss: 0.1395 - lr: 5.0000e-04
Epoch 9/50
1777/1777 [==============================] - 342s 192ms/step - loss: 0.1352 - lr: 5.0000e-04
Epoch 10/50
1777/1777 [=========================

In [ ]:
#importLibraries
from keras.models import load_model
from keras.models import Model
#from conf import myConfig as config
from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import BatchNormalization
from keras.layers import Activation
import cv2
import numpy as np
#from skimage.measure import compare_psnr
import argparse
from pathlib import Path
import keras.backend as K
from google.colab.patches import cv2_imshow
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio 

#ParsingArguments
weightsPath='/content/myModel_hybrid_layer.h5'
dataPath='/content/drive/MyDrive/original'

#createModel, loadWeights
def custom_loss(y_true,y_pred): #this is required for loading a keras-model created with custom-loss
    diff=y_true-y_pred
    res=K.sum(diff*diff)/(2*batch_size)
    return res
nmodel=load_model(weightsPath,custom_objects={'custom_loss':custom_loss})
print('nmodel is loaded')

#createArrayOfTestImages
p=Path(dataPath)
listPaths=list(p.glob('./*.jpg'))
imgTestArray = []
for path in listPaths:
    imgTestArray.append(((cv2.resize
    (cv2.imread(str(path),0),(200,200),
    interpolation=cv2.INTER_CUBIC))))
imgTestArray=np.array(imgTestArray)/255
    
#calculatePSNR
sumPSNR=0
for i in range(0,len(imgTestArray)):
    #cv2.imshow('trueCleanImage',imgTestArray[i]); cv2.waitKey(0)
    noisyImage=imgTestArray[i]+np.random.normal(0.0,25/255, 
        imgTestArray[i].shape)
   # cv2.imshow('noisyImage',noisyImage); cv2.waitKey(0)
        #print(np.expand_dims(np.expand_dims(noisyImage,axis=2),axis=0).shape)
    error=nmodel.predict(np.expand_dims(np.expand_dims(noisyImage,axis=2),axis=0))
    predClean=noisyImage-np.squeeze(error)
        #print(error.min(),error.max())
    #cv2.imshow('predCleanImage',predClean); cv2.waitKey(0)
    psnr=peak_signal_noise_ratio(imgTestArray[i],predClean)
    sumPSNR=sumPSNR+psnr
    cv2.destroyAllWindows()
avgPSNR=sumPSNR/len(imgTestArray)
print('avgPSNR on test-data',avgPSNR)
print(sumPSNR)

nmodel is loaded
1/1 [==============================] - 0s 20ms/step
avgPSNR on test-data 27.90723250506475
1506.9905552734965


In [ ]:
## model with conv2d, dialated conv2d and separable conv2d , second varient layer

from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Conv2D,SeparableConv2D
from keras.layers import BatchNormalization
from keras.layers import Activation
from keras import optimizers
import keras.backend as K
import numpy as np
from tensorflow import keras
import pandas as pd

# create CNN model
model=Sequential()

model.add(Conv2D(64,(3,3),padding="same",input_shape=(None,None,1),dilation_rate=2))
model.add(Activation('relu'))
#model.add(MultiResBlock(64, input))
for layers in range(2,7):
    model.add(Conv2D(64,(3,3),padding="same"))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv2D(64,(3,3),padding="same", dilation_rate=2))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(SeparableConv2D(64,(3,3),padding="same"))
    model.add(BatchNormalization())
    model.add(Activation('relu'))

model.add(Conv2D(1,(3,3),padding="same", dilation_rate=2))
model.summary()
 


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, None, None, 64)    640       
                                                                 
 activation (Activation)     (None, None, None, 64)    0         
                                                                 
 conv2d_1 (Conv2D)           (None, None, None, 64)    36928     
                                                                 
 batch_normalization (BatchN  (None, None, None, 64)   256       
 ormalization)                                                   
                                                                 
 activation_1 (Activation)   (None, None, None, 64)    0         
                                                                 
 conv2d_2 (Conv2D)           (None, None, None, 64)    36928     
                                                        

In [ ]:
# load the data and normalize it
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio 
import tensorflow as tf

cleanImages=np.load(data)
cleanImages=cleanImages/255.0
cleanImages=cleanImages.astype('float32')
 
 
 
 
# define augmentor and create custom flow
aug = ImageDataGenerator(rotation_range=30, fill_mode="nearest")
def myFlow(generator,X):
      for batch in generator.flow(x=X,batch_size=batch_size,seed=7):
        trueNoiseBatch=np.random.normal(0.0,10/255,batch.shape)
        noisyImagesBatch=batch+trueNoiseBatch
        yield (noisyImagesBatch,trueNoiseBatch)
 
 
# create custom loss, compile the model
print("[INFO] compilingTheModel")
opt=keras.optimizers.Adam(learning_rate=0.001)
def custom_loss(y_true,y_pred):
    diff=y_true-y_pred
    res=K.sum(diff*diff)/(2*batch_size)
    return res
model.compile(loss= custom_loss,optimizer=opt)
#
print(len(cleanImages))


# train
model.fit(myFlow(aug,cleanImages),epochs=epochs,steps_per_epoch=len(cleanImages)//batch_size,verbose=1)
# save the model
model.save('myModel_hybrid_layer_dilated_varient.h5')

[INFO] compilingTheModel
349150
Epoch 1/10
1363/1363 [==============================] - 686s 485ms/step - loss: 2.5036
Epoch 2/10
1363/1363 [==============================] - 667s 488ms/step - loss: 0.3919
Epoch 3/10
1363/1363 [==============================] - 665s 488ms/step - loss: 0.2418
Epoch 4/10
1363/1363 [==============================] - 665s 488ms/step - loss: 0.1732
Epoch 5/10
1363/1363 [==============================] - 665s 488ms/step - loss: 0.1455
Epoch 6/10
1363/1363 [==============================] - 665s 488ms/step - loss: 0.1463
Epoch 7/10
1363/1363 [==============================] - 664s 487ms/step - loss: 0.1219
Epoch 8/10
1363/1363 [==============================] - 664s 488ms/step - loss: 0.1190
Epoch 9/10
1363/1363 [==============================] - 665s 488ms/step - loss: 0.1061
Epoch 10/10
1363/1363 [==============================] - 665s 488ms/step - loss: 0.0973


In [ ]:
#importLibraries
from keras.models import load_model
from keras.models import Model
#from conf import myConfig as config
from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import BatchNormalization
from keras.layers import Activation
import cv2
import numpy as np
#from skimage.measure import compare_psnr
import argparse
from pathlib import Path
import keras.backend as K
from google.colab.patches import cv2_imshow
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio 

#ParsingArguments
weightsPath='/content/myModel_hybrid_layer_dilated_varient.h5'
dataPath='/content/drive/MyDrive/original'

#createModel, loadWeights
def custom_loss(y_true,y_pred): #this is required for loading a keras-model created with custom-loss
    diff=y_true-y_pred
    res=K.sum(diff*diff)/(2*batch_size)
    return res
nmodel=load_model(weightsPath,custom_objects={'custom_loss':custom_loss})
print('nmodel is loaded')

#createArrayOfTestImages
p=Path(dataPath)
listPaths=list(p.glob('./*.jpg'))
imgTestArray = []
for path in listPaths:
    imgTestArray.append(((cv2.resize
    (cv2.imread(str(path),0),(200,200),
    interpolation=cv2.INTER_CUBIC))))
imgTestArray=np.array(imgTestArray)/255
    
#calculatePSNR
sumPSNR=0
sumSSIM=0
for i in range(0,len(imgTestArray)):
    #cv2.imshow('trueCleanImage',imgTestArray[i]); cv2.waitKey(0)
    noisyImage=imgTestArray[i]+np.random.normal(0.0,25/255, 
        imgTestArray[i].shape)
   # cv2.imshow('noisyImage',noisyImage); cv2.waitKey(0)
        #print(np.expand_dims(np.expand_dims(noisyImage,axis=2),axis=0).shape)
    error=nmodel.predict(np.expand_dims(np.expand_dims(noisyImage,axis=2),axis=0))
    predClean=noisyImage-np.squeeze(error)
        #print(error.min(),error.max())
    #cv2.imshow('predCleanImage',predClean); cv2.waitKey(0)
    psnr=peak_signal_noise_ratio(imgTestArray[i],predClean)
    sumPSNR=sumPSNR+psnr
    SSIM= ssim(imgTestArray[i],predClean)
    sumSSIM=sumSSIM+SSIM
    #cv2.destroyAllWindows()
avgPSNR=sumPSNR/len(imgTestArray)
avgSSIM=sumSSIM/len(imgTestArray)
print('avgPSNR on test-data',avgPSNR)
print('avgSSIM on test-data',avgSSIM)
#print(sumPSNR)

nmodel is loaded
1/1 [==============================] - 0s 31ms/step
avgPSNR on test-data 26.81809600181865
avgSSIM on test-data 0.6972718101821781
